# 📊 Análisis: Dónde se Guarda la Información General

Este notebook analiza el flujo completo de datos desde la **información general** de una encuesta hasta su almacenamiento en la base de datos PostgreSQL.

## 🎯 Objetivo
Identificar **exactamente dónde** se almacena cada campo de la información general y cómo se estructura en las tablas de la base de datos.

## 1. Setup y Configuración

Configuramos las conexiones necesarias para analizar la base de datos.

In [ ]:
// Importar librerías necesarias para el análisis
import sequelize from '../src/config/sequelize.js';
import { QueryTypes } from 'sequelize';
import { Familias, Municipios, Parroquia, Sector, Veredas } from '../src/models/index.js';

console.log('✅ Librerías importadas correctamente');
console.log('📋 Modelos disponibles:', {
  Familias: !!Familias,
  Municipios: !!Municipios,
  Parroquia: !!Parroquia,
  Sector: !!Sector,
  Veredas: !!Veredas
});

## 2. Conexión a Base de Datos y Modelos

Verificamos la conexión y exploramos los modelos principales.

In [ ]:
// Verificar conexión a la base de datos
try {
  await sequelize.authenticate();
  console.log('✅ Conexión a PostgreSQL establecida exitosamente');
  
  // Obtener información de la base de datos
  const dbInfo = await sequelize.query(
    "SELECT current_database() as database, current_user as user, version() as version",
    { type: QueryTypes.SELECT }
  );
  
  console.log('📊 Información de la base de datos:', dbInfo[0]);
  
} catch (error) {
  console.error('❌ Error conectando a la base de datos:', error.message);
}

## 3. Esquema de Almacenamiento de Información General

Exploramos la estructura de la tabla principal donde se almacena la información general: **`familias`**

In [ ]:
// Analizar estructura de la tabla familias
const estructuraFamilias = await sequelize.query(
  `SELECT 
    column_name as campo,
    data_type as tipo,
    is_nullable as permite_null,
    column_default as valor_defecto
  FROM information_schema.columns 
  WHERE table_name = 'familias' 
  ORDER BY ordinal_position`,
  { type: QueryTypes.SELECT }
);

console.log('🏗️ ESTRUCTURA DE LA TABLA FAMILIAS:');
console.log('=' .repeat(60));

// Categorizar campos por tipo de información
const camposCategorizados = {
  identificacion: [],
  informacion_basica: [],
  geografica: [],
  contacto: [],
  encuesta: [],
  servicios: [],
  metadata: []
};

estructuraFamilias.forEach(campo => {
  const nombre = campo.campo;
  
  if (nombre.includes('id_familia')) {
    camposCategorizados.identificacion.push(campo);
  } else if (['apellido_familiar', 'direccion_familia', 'tamaño_familia'].includes(nombre)) {
    camposCategorizados.informacion_basica.push(campo);
  } else if (['id_municipio', 'id_vereda', 'id_sector', 'sector'].includes(nombre)) {
    camposCategorizados.geografica.push(campo);
  } else if (['telefono', 'email', 'numero_contacto'].includes(nombre)) {
    camposCategorizados.contacto.push(campo);
  } else if (['estado_encuesta', 'numero_encuestas', 'fecha_ultima_encuesta', 'codigo_familia'].includes(nombre)) {
    camposCategorizados.encuesta.push(campo);
  } else if (['tipo_vivienda', 'id_tipo_vivienda', 'comunionEnCasa'].includes(nombre)) {
    camposCategorizados.servicios.push(campo);
  } else {
    camposCategorizados.metadata.push(campo);
  }
});

// Mostrar campos categorizados
Object.entries(camposCategorizados).forEach(([categoria, campos]) => {
  if (campos.length > 0) {
    console.log(`\n📋 ${categoria.toUpperCase()}:`);
    campos.forEach(campo => {
      console.log(`  • ${campo.campo} (${campo.tipo}) - Null: ${campo.permite_null}`);
    });
  }
});

## 4. Almacenamiento de Información Familiar

Analizamos cómo se almacena la información básica de la familia en la tabla `familias`.

In [ ]:
// Ejemplo de mapeo: informacionGeneral -> tabla familias
const ejemploInformacionGeneral = {
  // Datos que llegan del frontend
  apellido_familiar: "Rodríguez García",
  direccion: "Carrera 45 # 23-67",
  telefono: "3001234567",
  email: "rodriguez.garcia@example.com",
  municipio: { id: 3, nombre: "Medellín" },
  vereda: { id: 1, nombre: "Centro" },
  sector: { id: 1, nombre: "Zona Centro" },
  comunionEnCasa: false
};

console.log('📥 EJEMPLO DE INFORMACIÓN GENERAL (Frontend):');
console.log(JSON.stringify(ejemploInformacionGeneral, null, 2));

// Mapeo a campos de base de datos
const mapeoBaseDatos = {
  "apellido_familiar": "familias.apellido_familiar",
  "direccion": "familias.direccion_familia", 
  "telefono": "familias.telefono",
  "email": "familias.email",
  "municipio.id": "familias.id_municipio",
  "vereda.id": "familias.id_vereda", 
  "sector.id": "familias.id_sector",
  "sector.nombre": "familias.sector (fallback)",
  "comunionEnCasa": "familias.comunionEnCasa"
};

console.log('\n🗂️ MAPEO A BASE DE DATOS:');
console.log('=' .repeat(50));
Object.entries(mapeoBaseDatos).forEach(([origen, destino]) => {
  console.log(`${origen} → ${destino}`);
});

## 5. Almacenamiento de Información Geográfica

Exploramos cómo se almacenan y relacionan los datos geográficos.

In [ ]:
// Analizar tablas geográficas y sus relaciones
const tablasGeograficas = ['municipios', 'veredas', 'sectores'];

for (const tabla of tablasGeograficas) {
  try {
    const estructura = await sequelize.query(
      `SELECT column_name, data_type, is_nullable 
       FROM information_schema.columns 
       WHERE table_name = '${tabla}' 
       ORDER BY ordinal_position`,
      { type: QueryTypes.SELECT }
    );
    
    const registros = await sequelize.query(
      `SELECT COUNT(*) as total FROM ${tabla}`,
      { type: QueryTypes.SELECT }
    );
    
    console.log(`\n🌍 TABLA: ${tabla.toUpperCase()}`);
    console.log(`📊 Total registros: ${registros[0].total}`);
    console.log('📋 Estructura:');
    estructura.forEach(col => {
      console.log(`  • ${col.column_name} (${col.data_type})`);
    });
    
  } catch (error) {
    console.log(`❌ Error analizando tabla ${tabla}: ${error.message}`);
  }
}

// Verificar relaciones en familias
console.log('\n🔗 RELACIONES GEOGRÁFICAS EN FAMILIAS:');
const relacionesGeograficas = await sequelize.query(
  `SELECT 
    COUNT(CASE WHEN id_municipio IS NOT NULL THEN 1 END) as con_municipio,
    COUNT(CASE WHEN id_vereda IS NOT NULL THEN 1 END) as con_vereda,
    COUNT(CASE WHEN id_sector IS NOT NULL THEN 1 END) as con_sector,
    COUNT(CASE WHEN sector IS NOT NULL THEN 1 END) as con_sector_texto,
    COUNT(*) as total_familias
  FROM familias`,
  { type: QueryTypes.SELECT }
);

console.log('📊 Estadísticas de relaciones geográficas:');
console.log(relacionesGeograficas[0]);

## 6. Almacenamiento de Registro de Servicios

Analizamos las tablas relacionadas donde se almacenan los servicios (agua, basuras, vivienda).

In [ ]:
// Analizar tablas de servicios relacionadas
const tablasServicios = [
  'familia_disposicion_basura',
  'familia_sistema_acueducto', 
  'familia_sistema_aguas_residuales',
  'familia_tipo_vivienda'
];

console.log('🏠 ANÁLISIS DE TABLAS DE SERVICIOS:');
console.log('=' .repeat(60));

for (const tabla of tablasServicios) {
  try {
    // Estructura de la tabla
    const estructura = await sequelize.query(
      `SELECT column_name, data_type, is_nullable 
       FROM information_schema.columns 
       WHERE table_name = '${tabla}' 
       ORDER BY ordinal_position`,
      { type: QueryTypes.SELECT }
    );
    
    // Conteo de registros
    const registros = await sequelize.query(
      `SELECT COUNT(*) as total FROM ${tabla}`,
      { type: QueryTypes.SELECT }
    );
    
    console.log(`\n🔧 ${tabla}:`);
    console.log(`📊 Registros: ${registros[0].total}`);
    console.log('📋 Campos:');
    estructura.forEach(col => {
      console.log(`  • ${col.column_name} (${col.data_type})`);
    });
    
  } catch (error) {
    console.log(`❌ Tabla ${tabla} no encontrada o error: ${error.message}`);
  }
}

// Analizar tablas de catálogos de servicios
console.log('\n📚 CATÁLOGOS DE SERVICIOS:');
const catalogosServicios = [
  'tipos_disposicion_basura',
  'sistemas_acueducto',
  'tipos_aguas_residuales',
  'tipos_vivienda'
];

for (const catalogo of catalogosServicios) {
  try {
    const datos = await sequelize.query(
      `SELECT COUNT(*) as total FROM ${catalogo}`,
      { type: QueryTypes.SELECT }
    );
    
    const muestra = await sequelize.query(
      `SELECT * FROM ${catalogo} LIMIT 3`,
      { type: QueryTypes.SELECT }
    );
    
    console.log(`\n📖 ${catalogo}:`);
    console.log(`📊 Total opciones: ${datos[0].total}`);
    console.log('🔍 Muestra:', muestra);
    
  } catch (error) {
    console.log(`❌ Catálogo ${catalogo} no encontrado: ${error.message}`);
  }
}

## 7. Flujo Completo: Del Frontend a la Base de Datos

Trazamos el flujo completo de cómo la información general viaja desde el frontend hasta su almacenamiento final.

In [ ]:
// Demostrar el flujo completo con un ejemplo real
const flujoCompleto = {
  "1_frontend_request": {
    "informacionGeneral": {
      "apellido_familiar": "Rodríguez García",
      "direccion": "Carrera 45 # 23-67",
      "telefono": "3001234567",
      "municipio": { "id": "3", "nombre": "Medellín" },
      "vereda": { "id": "1", "nombre": "Centro" },
      "sector": { "id": "1", "nombre": "Zona Centro" },
      "comunionEnCasa": false
    }
  },
  
  "2_controller_processing": {
    "function": "crearEncuesta()",
    "extraction": "const { informacionGeneral } = req.body",
    "mapping": {
      "apellido_familiar": "informacionGeneral.apellido_familiar",
      "direccion_familia": "informacionGeneral.direccion", 
      "telefono": "informacionGeneral.telefono",
      "id_municipio": "parseInt(informacionGeneral.municipio.id)",
      "id_vereda": "parseInt(informacionGeneral.vereda.id)",
      "id_sector": "parseInt(informacionGeneral.sector.id)",
      "sector": "informacionGeneral.sector.nombre (fallback)",
      "comunionEnCasa": "informacionGeneral.comunionEnCasa || false"
    }
  },
  
  "3_database_insert": {
    "tabla_principal": "familias",
    "campos_insertados": [
      "apellido_familiar",
      "direccion_familia", 
      "telefono",
      "id_municipio",
      "id_vereda", 
      "id_sector",
      "sector",
      "comunionEnCasa",
      "estado_encuesta",
      "fecha_ultima_encuesta",
      "codigo_familia"
    ],
    "metodo": "await Familias.create(familiaData, { transaction })"
  },
  
  "4_related_tables": {
    "servicios_insertados": [
      "familia_disposicion_basura (basado en vivienda.disposicion_basuras)",
      "familia_sistema_acueducto (basado en servicios_agua.sistema_acueducto)", 
      "familia_sistema_aguas_residuales (basado en servicios_agua.aguas_residuales)",
      "familia_tipo_vivienda (basado en vivienda.tipo_vivienda)"
    ]
  }
};

console.log('🔄 FLUJO COMPLETO DE DATOS:');
console.log('=' .repeat(80));
console.log(JSON.stringify(flujoCompleto, null, 2));

// Verificar con un ejemplo real de la base de datos
console.log('\n✅ VERIFICACIÓN CON DATOS REALES:');
const ejemploReal = await sequelize.query(
  `SELECT 
    id_familia,
    apellido_familiar,
    direccion_familia,
    telefono,
    id_municipio,
    id_vereda,
    id_sector,
    sector,
    "comunionEnCasa",
    estado_encuesta,
    fecha_ultima_encuesta
  FROM familias 
  LIMIT 1`,
  { type: QueryTypes.SELECT }
);

if (ejemploReal.length > 0) {
  console.log('📋 Ejemplo real de familia almacenada:');
  console.log(ejemploReal[0]);
} else {
  console.log('❌ No hay familias en la base de datos para mostrar como ejemplo');
}

## 8. Resumen y Conclusiones

Resumen de dónde se almacena cada componente de la información general.

In [ ]:
// Resumen completo del almacenamiento
const resumenAlmacenamiento = {
  "INFORMACIÓN_GENERAL": {
    "ubicacion_principal": "tabla 'familias'",
    "campos_directos": {
      "apellido_familiar": "familias.apellido_familiar",
      "direccion": "familias.direccion_familia",
      "telefono": "familias.telefono", 
      "email": "familias.email",
      "comunionEnCasa": "familias.comunionEnCasa"
    },
    "campos_geograficos": {
      "municipio_id": "familias.id_municipio → municipios.id_municipio",
      "vereda_id": "familias.id_vereda → veredas.id_vereda",
      "sector_id": "familias.id_sector → sectores.id_sector",
      "sector_texto": "familias.sector (fallback cuando no hay ID)"
    },
    "metadatos_automaticos": {
      "codigo_familia": "generado automáticamente",
      "estado_encuesta": "'completed' por defecto", 
      "fecha_ultima_encuesta": "fecha actual",
      "numero_encuestas": "1 por defecto",
      "tamaño_familia": "calculado basado en miembros"
    }
  },
  
  "SERVICIOS_RELACIONADOS": {
    "disposicion_basuras": "familia_disposicion_basura (tabla puente)",
    "sistema_acueducto": "familia_sistema_acueducto (tabla puente)", 
    "aguas_residuales": "familia_sistema_aguas_residuales (tabla puente)",
    "tipo_vivienda": "familia_tipo_vivienda (tabla puente)"
  },
  
  "DATOS_PERSONAS": {
    "miembros_vivos": "tabla 'personas' (id_familia_familias)",
    "miembros_fallecidos": "tabla 'personas' (identificacion LIKE 'FALLECIDO%')",
    "informacion_heredada": [
      "primer_apellido = informacionGeneral.apellido_familiar",
      "direccion = informacionGeneral.direccion",
      "telefono = informacionGeneral.telefono (si no tienen propio)"
    ]
  }
};

console.log('📊 RESUMEN COMPLETO DEL ALMACENAMIENTO:');
console.log('=' .repeat(80));
console.log(JSON.stringify(resumenAlmacenamiento, null, 2));

console.log('\n✅ PUNTOS CLAVE:');
console.log('• La información general se almacena PRINCIPALMENTE en la tabla "familias"');
console.log('• Los datos geográficos usan IDs de referencia a tablas de catálogo');
console.log('• Los servicios se almacenan en tablas de relación (familia_* patterns)');
console.log('• La información se hereda a las personas miembro de la familia');
console.log('• Se generan metadatos automáticamente (códigos, fechas, estados)');

console.log('\n🎯 TABLA PRINCIPAL: familias');
console.log('🔗 TABLAS RELACIONADAS: familia_disposicion_basura, familia_sistema_acueducto, etc.');
console.log('👥 TABLA DEPENDIENTE: personas (hereda información familiar)');